In [1]:
from pathlib import Path
import pandas as pd

# ============================================================
# 02.17 STEP 1 — TARGET & ACTUAL DATA INSPECTION
# ============================================================

TARGET_FILE = Path("../data/synthetic/targets.csv")
SALES_DIR = Path("../data/synthetic")

# ------------------------------------------------------------
# Load target data
# ------------------------------------------------------------

targets = pd.read_csv(TARGET_FILE)

print("02.17 STEP 1 — TARGET & ACTUAL DATA INSPECTION")
print("=" * 65)

print("\nTarget Data")
print("-" * 65)

print("Target rows:", len(targets))
print("Target columns:")
print(targets.columns.tolist())

print("\nTarget date range:")
print("From:", targets["target_month"].min())
print("To:", targets["target_month"].max())

print("\nUnique stores:", targets["store_id"].nunique())
print("Unique categories:", targets["category"].nunique())

# ------------------------------------------------------------
# Load actual sales files
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)

    df["order_date"] = pd.to_datetime(df["order_date"])

    df["gross_sales_value"] = (
        df["quantity"] * df["unit_selling_price"]
    )

    df["discount_value"] = (
        df["gross_sales_value"] * df["discount_pct"] / 100
    )

    df["net_sales_value"] = (
        df["gross_sales_value"] - df["discount_value"]
    )

    sales_list.append(
        df[
            [
                "order_date",
                "store_id",
                "product_id",
                "quantity",
                "net_sales_value"
            ]
        ]
    )

sales = pd.concat(
    sales_list,
    ignore_index=True
)

# ------------------------------------------------------------
# Create monthly actual sales
# ------------------------------------------------------------

sales["target_month"] = (
    sales["order_date"]
    .dt.to_period("M")
    .astype(str)
)

actual_monthly = (
    sales
    .groupby(
        ["target_month", "store_id"],
        as_index=False
    )
    .agg(
        actual_revenue=("net_sales_value", "sum"),
        actual_orders=("order_date", "count")
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nActual Sales Data")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Sales rows:", len(sales))

print(
    "Actual sales months:",
    actual_monthly["target_month"].nunique()
)

print(
    "Actual stores:",
    actual_monthly["store_id"].nunique()
)

print("\nTarget Sample")
print("-" * 65)

print(targets.head())

print("\nActual Monthly Sample")
print("-" * 65)

print(actual_monthly.head())

# ------------------------------------------------------------
# Final inspection validation
# ------------------------------------------------------------

required_target_columns = [
    "target_month",
    "store_id",
    "category",
    "revenue_target",
    "profit_target",
    "order_target",
    "return_rate_target_pct"
]

missing_target_columns = [
    col
    for col in required_target_columns
    if col not in targets.columns
]

print("\nValidation")
print("-" * 65)

print(
    "Required target columns missing:",
    missing_target_columns
)

print(
    "Target data available:",
    len(targets) > 0
)

print(
    "Actual sales data available:",
    len(actual_monthly) > 0
)

if (
    len(targets) > 0
    and len(actual_monthly) > 0
    and len(missing_target_columns) == 0
):
    print("\n02.17 Step 1 inspection: PASSED")
else:
    print("\n02.17 Step 1 inspection: FAILED")

02.17 STEP 1 — TARGET & ACTUAL DATA INSPECTION

Target Data
-----------------------------------------------------------------
Target rows: 8980
Target columns:
['target_id', 'target_month', 'store_id', 'category', 'revenue_target', 'profit_target', 'order_target', 'return_rate_target_pct']

Target date range:
From: 2024-01-01
To: 2026-08-01

Unique stores: 60
Unique categories: 5

Actual Sales Data
-----------------------------------------------------------------
Sales files: 32
Sales rows: 2118200
Actual sales months: 32
Actual stores: 60

Target Sample
-----------------------------------------------------------------
            target_id target_month    store_id                category  \
0  TGT20240100001B&PC   2024-01-01  STORE00001  Beauty & Personal Care   
1     TGT20240100001E   2024-01-01  STORE00001             Electronics   
2     TGT20240100001F   2024-01-01  STORE00001                 Fashion   
3     TGT20240100001G   2024-01-01  STORE00001                 Grocery   
4  

In [2]:
# ============================================================
# 02.17 STEP 2 — PREVIOUS-PERIOD ACTUAL PERFORMANCE
# ============================================================

# Load product master to connect product_id with category
PRODUCT_FILE = Path("../data/synthetic/products_master.csv")

products = pd.read_csv(PRODUCT_FILE)

# Keep only required product-category relationship
product_category = products[
    ["product_id", "category"]
].drop_duplicates()

# ------------------------------------------------------------
# Add category to actual sales
# ------------------------------------------------------------

sales_with_category = sales.merge(
    product_category,
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# Monthly actual performance
# Store + Category + Month
# ------------------------------------------------------------

actual_monthly_category = (
    sales_with_category
    .groupby(
        ["target_month", "store_id", "category"],
        as_index=False
    )
    .agg(
        actual_revenue=("net_sales_value", "sum"),
        actual_orders=("order_date", "count")
    )
)

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

actual_monthly_category["target_month"] = pd.to_datetime(
    actual_monthly_category["target_month"]
)

actual_monthly_category = actual_monthly_category.sort_values(
    ["store_id", "category", "target_month"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Previous-period performance
# ------------------------------------------------------------

actual_monthly_category["previous_month"] = (
    actual_monthly_category
    .groupby(["store_id", "category"])["target_month"]
    .shift(1)
)

actual_monthly_category["previous_actual_revenue"] = (
    actual_monthly_category
    .groupby(["store_id", "category"])["actual_revenue"]
    .shift(1)
)

actual_monthly_category["previous_actual_orders"] = (
    actual_monthly_category
    .groupby(["store_id", "category"])["actual_orders"]
    .shift(1)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("02.17 STEP 2 — PREVIOUS-PERIOD ACTUAL PERFORMANCE")
print("=" * 65)

print("\nActual Store-Category-Month Data")
print("-" * 65)

print(
    "Rows:",
    len(actual_monthly_category)
)

print(
    "Stores:",
    actual_monthly_category["store_id"].nunique()
)

print(
    "Categories:",
    actual_monthly_category["category"].nunique()
)

print(
    "Months:",
    actual_monthly_category["target_month"].nunique()
)

print("\nSample")
print("-" * 65)

print(
    actual_monthly_category[
        [
            "target_month",
            "store_id",
            "category",
            "actual_revenue",
            "actual_orders",
            "previous_actual_revenue",
            "previous_actual_orders"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# Check previous-period records
# ------------------------------------------------------------

previous_period_available = (
    actual_monthly_category["previous_actual_revenue"]
    .notna()
    .sum()
)

print("\nPrevious-Period Validation")
print("-" * 65)

print(
    "Rows with previous-period revenue:",
    previous_period_available
)

print(
    "Rows without previous-period revenue:",
    actual_monthly_category[
        "previous_actual_revenue"
    ].isna().sum()
)

# First month for each store/category naturally has no previous month.
first_month = actual_monthly_category["target_month"].min()

first_month_missing = actual_monthly_category.loc[
    actual_monthly_category["target_month"] == first_month,
    "previous_actual_revenue"
].isna().all()

print(
    "First month correctly has no previous-period value:",
    first_month_missing
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

validation_passed = (
    len(actual_monthly_category) > 0
    and actual_monthly_category["store_id"].nunique() == 60
    and actual_monthly_category["category"].nunique() == 5
    and actual_monthly_category["target_month"].nunique() == 32
    and first_month_missing
)

if validation_passed:
    print("\n02.17 Step 2 validation: PASSED")
else:
    print("\n02.17 Step 2 validation: FAILED")

02.17 STEP 2 — PREVIOUS-PERIOD ACTUAL PERFORMANCE

Actual Store-Category-Month Data
-----------------------------------------------------------------
Rows: 8980
Stores: 60
Categories: 5
Months: 32

Sample
-----------------------------------------------------------------
  target_month    store_id                category  actual_revenue  \
0   2024-01-01  STORE00001  Beauty & Personal Care     10656030.58   
1   2024-02-01  STORE00001  Beauty & Personal Care      9516267.42   
2   2024-03-01  STORE00001  Beauty & Personal Care     10398969.17   
3   2024-04-01  STORE00001  Beauty & Personal Care     11332950.00   
4   2024-05-01  STORE00001  Beauty & Personal Care     12524282.42   
5   2024-06-01  STORE00001  Beauty & Personal Care     12889925.46   
6   2024-07-01  STORE00001  Beauty & Personal Care     10745393.31   
7   2024-08-01  STORE00001  Beauty & Personal Care     13097629.73   
8   2024-09-01  STORE00001  Beauty & Personal Care     12134049.75   
9   2024-10-01  STORE00001  B

In [3]:
# ============================================================
# 02.17 STEP 3 — PLANNED GROWTH LOGIC
# ============================================================

# Planned growth inputs
# These are parameters, not hard-coded business results.
# Update these values only when the approved simulation plan
# provides the growth assumptions.

PLANNED_REVENUE_GROWTH_PCT = 5.0
PLANNED_ORDER_GROWTH_PCT = 5.0

print("02.17 STEP 3 — PLANNED GROWTH LOGIC")
print("=" * 65)

print("\nPlanned Growth Parameters")
print("-" * 65)

print(
    "Revenue growth:",
    PLANNED_REVENUE_GROWTH_PCT,
    "%"
)

print(
    "Order growth:",
    PLANNED_ORDER_GROWTH_PCT,
    "%"
)

# ------------------------------------------------------------
# Calculate planned targets from previous-period actuals
# ------------------------------------------------------------

growth_plan = actual_monthly_category.copy()

growth_plan["planned_revenue_target"] = (
    growth_plan["previous_actual_revenue"]
    * (1 + PLANNED_REVENUE_GROWTH_PCT / 100)
)

growth_plan["planned_order_target"] = (
    growth_plan["previous_actual_orders"]
    * (1 + PLANNED_ORDER_GROWTH_PCT / 100)
)

# Round order target to whole orders
growth_plan["planned_order_target"] = (
    growth_plan["planned_order_target"]
    .round()
    .astype("Int64")
)

# ------------------------------------------------------------
# First month cannot use previous-period actual
# ------------------------------------------------------------

growth_plan["growth_base_available"] = (
    growth_plan["previous_actual_revenue"].notna()
    & growth_plan["previous_actual_orders"].notna()
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nGrowth Plan Sample")
print("-" * 65)

print(
    growth_plan[
        [
            "target_month",
            "store_id",
            "category",
            "previous_actual_revenue",
            "planned_revenue_target",
            "previous_actual_orders",
            "planned_order_target",
            "growth_base_available"
        ]
    ].head(10)
)

print("\nValidation")
print("-" * 65)

print(
    "Revenue growth rate valid:",
    PLANNED_REVENUE_GROWTH_PCT >= 0
)

print(
    "Order growth rate valid:",
    PLANNED_ORDER_GROWTH_PCT >= 0
)

print(
    "Growth base available:",
    growth_plan["growth_base_available"].sum()
)

print(
    "Rows without growth base:",
    (~growth_plan["growth_base_available"]).sum()
)

# ------------------------------------------------------------
# Check target relationship
# ------------------------------------------------------------

valid_growth_rows = growth_plan[
    growth_plan["growth_base_available"]
].copy()

revenue_relationship_valid = (
    valid_growth_rows["planned_revenue_target"]
    >= valid_growth_rows["previous_actual_revenue"]
).all()

order_relationship_valid = (
    valid_growth_rows["planned_order_target"]
    >= valid_growth_rows["previous_actual_orders"]
).all()

print(
    "Revenue target follows previous actual + growth:",
    revenue_relationship_valid
)

print(
    "Order target follows previous actual + growth:",
    order_relationship_valid
)

if (
    PLANNED_REVENUE_GROWTH_PCT >= 0
    and PLANNED_ORDER_GROWTH_PCT >= 0
    and revenue_relationship_valid
    and order_relationship_valid
):
    print("\n02.17 Step 3 validation: PASSED")
else:
    print("\n02.17 Step 3 validation: FAILED")

02.17 STEP 3 — PLANNED GROWTH LOGIC

Planned Growth Parameters
-----------------------------------------------------------------
Revenue growth: 5.0 %
Order growth: 5.0 %

Growth Plan Sample
-----------------------------------------------------------------
  target_month    store_id                category  previous_actual_revenue  \
0   2024-01-01  STORE00001  Beauty & Personal Care                      NaN   
1   2024-02-01  STORE00001  Beauty & Personal Care              10656030.58   
2   2024-03-01  STORE00001  Beauty & Personal Care               9516267.42   
3   2024-04-01  STORE00001  Beauty & Personal Care              10398969.17   
4   2024-05-01  STORE00001  Beauty & Personal Care              11332950.00   
5   2024-06-01  STORE00001  Beauty & Personal Care              12524282.42   
6   2024-07-01  STORE00001  Beauty & Personal Care              12889925.46   
7   2024-08-01  STORE00001  Beauty & Personal Care              10745393.31   
8   2024-09-01  STORE00001  Beau

In [6]:
# ============================================================
# 02.17 STEP 4 — CONTINUE GROWTH-BASED TARGET GENERATION
# ============================================================

print("02.17 STEP 4 — GROWTH-BASED TARGET CALCULATION")
print("=" * 65)

# Check that Step 3 output exists
print("\nStep 3 inputs")
print("-" * 65)

print("Revenue growth:", PLANNED_REVENUE_GROWTH_PCT, "%")
print("Order growth:", PLANNED_ORDER_GROWTH_PCT, "%")

# ------------------------------------------------------------
# Merge previous-period actuals from Step 2
# ------------------------------------------------------------

target_keys = [
    "target_month",
    "store_id",
    "category"
]

target_growth = targets.merge(
    growth_plan[
        [
            "target_month",
            "store_id",
            "category",
            "previous_actual_revenue",
            "previous_actual_orders",
            "growth_base_available"
        ]
    ],
    on=target_keys,
    how="left"
)

# ------------------------------------------------------------
# Calculate growth-based revenue target
# ------------------------------------------------------------

target_growth["growth_revenue_target"] = (
    target_growth["previous_actual_revenue"]
    * (1 + PLANNED_REVENUE_GROWTH_PCT / 100)
)

# ------------------------------------------------------------
# Calculate growth-based order target
# ------------------------------------------------------------

target_growth["growth_order_target"] = (
    target_growth["previous_actual_orders"]
    * (1 + PLANNED_ORDER_GROWTH_PCT / 100)
).round()

# ------------------------------------------------------------
# Keep existing first-month targets where previous
# period actual is unavailable
# ------------------------------------------------------------

target_growth["final_revenue_target"] = (
    target_growth["growth_revenue_target"]
    .fillna(target_growth["revenue_target"])
)

target_growth["final_order_target"] = (
    target_growth["growth_order_target"]
    .fillna(target_growth["order_target"])
)

# ------------------------------------------------------------
# Update target values
# ------------------------------------------------------------

target_growth["revenue_target"] = (
    target_growth["final_revenue_target"]
    .round(2)
)

target_growth["order_target"] = (
    target_growth["final_order_target"]
    .round()
    .astype(int)
)

# ------------------------------------------------------------
# Display sample
# ------------------------------------------------------------

print("\nGrowth-Based Target Sample")
print("-" * 65)

print(
    target_growth[
        [
            "target_month",
            "store_id",
            "category",
            "previous_actual_revenue",
            "revenue_target",
            "previous_actual_orders",
            "order_target"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

growth_rows = target_growth[
    target_growth["growth_base_available"]
].copy()

revenue_check = (
    growth_rows["revenue_target"]
    >= growth_rows["previous_actual_revenue"]
).all()

order_check = (
    growth_rows["order_target"]
    >= growth_rows["previous_actual_orders"]
).all()

print("\nValidation")
print("-" * 65)

print("Target rows:", len(target_growth))

print(
    "Missing revenue targets:",
    target_growth["revenue_target"].isna().sum()
)

print(
    "Missing order targets:",
    target_growth["order_target"].isna().sum()
)

print(
    "Negative revenue targets:",
    (target_growth["revenue_target"] < 0).sum()
)

print(
    "Negative order targets:",
    (target_growth["order_target"] < 0).sum()
)

print(
    "Revenue target >= previous actual:",
    revenue_check
)

print(
    "Order target >= previous actual:",
    order_check
)

if (
    target_growth["revenue_target"].notna().all()
    and target_growth["order_target"].notna().all()
    and (target_growth["revenue_target"] >= 0).all()
    and (target_growth["order_target"] >= 0).all()
    and revenue_check
    and order_check
):
    print("\n02.17 Step 4 validation: PASSED")
else:
    print("\n02.17 Step 4 validation: FAILED")

02.17 STEP 4 — GROWTH-BASED TARGET CALCULATION

Step 3 inputs
-----------------------------------------------------------------
Revenue growth: 5.0 %
Order growth: 5.0 %

Growth-Based Target Sample
-----------------------------------------------------------------
  target_month    store_id                category  previous_actual_revenue  \
0   2024-01-01  STORE00001  Beauty & Personal Care                      NaN   
1   2024-01-01  STORE00001             Electronics                      NaN   
2   2024-01-01  STORE00001                 Fashion                      NaN   
3   2024-01-01  STORE00001                 Grocery                      NaN   
4   2024-01-01  STORE00001          Home & Kitchen                      NaN   
5   2024-01-01  STORE00002  Beauty & Personal Care                      NaN   
6   2024-01-01  STORE00002             Electronics                      NaN   
7   2024-01-01  STORE00002                 Fashion                      NaN   
8   2024-01-01  STORE0000

In [7]:
# ============================================================
# 02.17 STEP 5 — SAVE GROWTH-BASED TARGET DATA
# ============================================================

from pathlib import Path

print("02.17 STEP 5 — SAVE GROWTH-BASED TARGET DATA")
print("=" * 65)

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUTPUT_DIR = BASE_DIR / "data" / "synthetic" / "target_growth_adjusted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "targets_growth_based.csv"

# ------------------------------------------------------------
# Select final target columns
# ------------------------------------------------------------

final_target_columns = [
    "target_id",
    "target_month",
    "store_id",
    "category",
    "revenue_target",
    "profit_target",
    "order_target",
    "return_rate_target_pct"
]

final_targets = target_growth[final_target_columns].copy()

# ------------------------------------------------------------
# Final data cleanup
# ------------------------------------------------------------

final_targets["target_month"] = pd.to_datetime(
    final_targets["target_month"]
)

final_targets["revenue_target"] = (
    final_targets["revenue_target"].round(2)
)

final_targets["profit_target"] = (
    final_targets["profit_target"].round(2)
)

final_targets["order_target"] = (
    final_targets["order_target"]
    .round()
    .astype(int)
)

final_targets["return_rate_target_pct"] = (
    final_targets["return_rate_target_pct"].round(2)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

final_targets.to_csv(
    OUTPUT_FILE,
    index=False
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nFinal Target Data")
print("-" * 65)

print("Rows:", len(final_targets))
print("Columns:", list(final_targets.columns))
print("Output file:", OUTPUT_FILE)

print("\nValidation")
print("-" * 65)

print(
    "Duplicate target IDs:",
    final_targets["target_id"].duplicated().sum()
)

print(
    "Missing revenue targets:",
    final_targets["revenue_target"].isna().sum()
)

print(
    "Missing profit targets:",
    final_targets["profit_target"].isna().sum()
)

print(
    "Missing order targets:",
    final_targets["order_target"].isna().sum()
)

print(
    "Negative revenue targets:",
    (final_targets["revenue_target"] < 0).sum()
)

print(
    "Negative order targets:",
    (final_targets["order_target"] < 0).sum()
)

if (
    len(final_targets) == 8980
    and final_targets["target_id"].duplicated().sum() == 0
    and final_targets["revenue_target"].notna().all()
    and final_targets["profit_target"].notna().all()
    and final_targets["order_target"].notna().all()
    and (final_targets["revenue_target"] >= 0).all()
    and (final_targets["order_target"] >= 0).all()
):
    print("\n02.17 Step 5 validation: PASSED")
else:
    print("\n02.17 Step 5 validation: FAILED")

02.17 STEP 5 — SAVE GROWTH-BASED TARGET DATA

Final Target Data
-----------------------------------------------------------------
Rows: 8980
Columns: ['target_id', 'target_month', 'store_id', 'category', 'revenue_target', 'profit_target', 'order_target', 'return_rate_target_pct']
Output file: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\target_growth_adjusted\targets_growth_based.csv

Validation
-----------------------------------------------------------------
Duplicate target IDs: 0
Missing revenue targets: 0
Missing profit targets: 0
Missing order targets: 0
Negative revenue targets: 0
Negative order targets: 0

02.17 Step 5 validation: PASSED


In [8]:
from pathlib import Path
import pandas as pd

print("02.17 STEP 6 - FINAL TARGET GROWTH VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

TARGET_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "target_growth_adjusted"
    / "targets_growth_based.csv"
)

targets = pd.read_csv(TARGET_FILE)

targets["target_month"] = pd.to_datetime(targets["target_month"])

print("\nTarget Data")
print("-" * 65)
print("Rows:", len(targets))
print("Columns:", list(targets.columns))
print(
    "Date range:",
    targets["target_month"].min().date(),
    "to",
    targets["target_month"].max().date()
)

# ---------------------------------------------------------
# Required column validation
# ---------------------------------------------------------

required_columns = [
    "target_id",
    "target_month",
    "store_id",
    "category",
    "revenue_target",
    "profit_target",
    "order_target",
    "return_rate_target_pct"
]

missing_columns = [
    col for col in required_columns
    if col not in targets.columns
]

# ---------------------------------------------------------
# Duplicate validation
# ---------------------------------------------------------

duplicate_target_ids = targets["target_id"].duplicated().sum()

duplicate_grain = targets.duplicated(
    subset=["target_month", "store_id", "category"]
).sum()

# ---------------------------------------------------------
# Missing-value validation
# ---------------------------------------------------------

missing_revenue = targets["revenue_target"].isna().sum()
missing_profit = targets["profit_target"].isna().sum()
missing_orders = targets["order_target"].isna().sum()

# ---------------------------------------------------------
# Value validation
# ---------------------------------------------------------

negative_revenue = (targets["revenue_target"] < 0).sum()
negative_profit = (targets["profit_target"] < 0).sum()
negative_orders = (targets["order_target"] < 0).sum()

invalid_return_rate = (
    (targets["return_rate_target_pct"] < 0)
    | (targets["return_rate_target_pct"] > 100)
).sum()

# ---------------------------------------------------------
# Growth-based target usefulness
# ---------------------------------------------------------

positive_revenue_targets = (
    targets["revenue_target"] > 0
).all()

positive_profit_targets = (
    targets["profit_target"] > 0
).all()

positive_order_targets = (
    targets["order_target"] > 0
).all()

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

validation_passed = (
    len(missing_columns) == 0
    and duplicate_target_ids == 0
    and duplicate_grain == 0
    and missing_revenue == 0
    and missing_profit == 0
    and missing_orders == 0
    and negative_revenue == 0
    and negative_profit == 0
    and negative_orders == 0
    and invalid_return_rate == 0
    and positive_revenue_targets
    and positive_profit_targets
    and positive_order_targets
)

print("\nValidation Results")
print("-" * 65)

print("Missing required columns:", missing_columns)
print("Duplicate target IDs:", duplicate_target_ids)
print("Duplicate month-store-category rows:", duplicate_grain)

print("Missing revenue targets:", missing_revenue)
print("Missing profit targets:", missing_profit)
print("Missing order targets:", missing_orders)

print("Negative revenue targets:", negative_revenue)
print("Negative profit targets:", negative_profit)
print("Negative order targets:", negative_orders)

print("Invalid return-rate targets:", invalid_return_rate)

print("Positive revenue targets:", positive_revenue_targets)
print("Positive profit targets:", positive_profit_targets)
print("Positive order targets:", positive_order_targets)

if validation_passed:
    print("\n02.17 Step 6 validation: PASSED")
else:
    print("\n02.17 Step 6 validation: FAILED")

02.17 STEP 6 - FINAL TARGET GROWTH VALIDATION

Target Data
-----------------------------------------------------------------
Rows: 8980
Columns: ['target_id', 'target_month', 'store_id', 'category', 'revenue_target', 'profit_target', 'order_target', 'return_rate_target_pct']
Date range: 2024-01-01 to 2026-08-01

Validation Results
-----------------------------------------------------------------
Missing required columns: []
Duplicate target IDs: 0
Duplicate month-store-category rows: 0
Missing revenue targets: 0
Missing profit targets: 0
Missing order targets: 0
Negative revenue targets: 0
Negative profit targets: 0
Negative order targets: 0
Invalid return-rate targets: 0
Positive revenue targets: True
Positive profit targets: True
Positive order targets: True

02.17 Step 6 validation: PASSED
